In [2]:
import os
import numpy as np
import pandas as pd
import pathlib as plt
import matplotlib.pyplot as plb
import tensorflow as tf
import cv2
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from tensorflow import keras
from keras import Model
from keras.layers import Input, Flatten, Dense, Conv2D, Convolution2D, Conv2DTranspose, MaxPooling2D, BatchNormalization, Activation, Concatenate, concatenate
from cv2 import imread, imwrite, imshow, resize
from pandas import DataFrame
from numpy import arange, array, random, zeros, ones, stack, argmax

In [3]:
train_images_names_path = r"C:\Users\Lenovo\Desktop\Courses\Programming\Programming docs\Datasets\YOLO-segmentation-dataset\aeroscapes\ImageSets\trn.txt"
test_images_names_path = r"C:\Users\Lenovo\Desktop\Courses\Programming\Programming docs\Datasets\YOLO-segmentation-dataset\aeroscapes\ImageSets\val.txt"
actual_images_directory_path = r"C:\Users\Lenovo\Desktop\Courses\Programming\Programming docs\Datasets\YOLO-segmentation-dataset\aeroscapes\JPEGImages"
masked_images_directory_path = r"C:\Users\Lenovo\Desktop\Courses\Programming\Programming docs\Datasets\YOLO-segmentation-dataset\aeroscapes\SegmentationClass"
actual_images_extention = ".jpg"
masked_images_extention = ".png"
num_classes = -1
image_size = 128
dataset_size = -1
batch_size = 16
epochs = 24

In [4]:
def conv_block(x, filters_count):
    x = Conv2D(filters_count, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2D(filters_count, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    return x

In [5]:
with open(train_images_names_path, 'r') as f: images_names = f.readlines()
with open(test_images_names_path, 'r') as f: images_names.extend(f.readlines())

In [6]:
df = DataFrame([
    (
        resize(imread(f"{actual_images_directory_path}\{file_name}{actual_images_extention}".replace('\n', '')), (image_size, image_size)),
        resize(imread(f"{masked_images_directory_path}\{file_name}{masked_images_extention}".replace('\n', '')), (image_size, image_size))[:,:,0]
    ) 
    for file_name 
    in images_names],
    columns=['image','mask'])

In [7]:
dataset_size = len(df)
num_classes = stack(df['mask']).flatten().max() + 1

In [8]:
sample_df = df[:100]

In [9]:
sample_df = sample_df/255.0

X_train, X_test, y_train, y_test = train_test_split(sample_df['image'], sample_df['mask'], test_size=0.15)
X_train, X_test, y_train, y_test = stack(X_train), stack(X_test), stack(y_train), stack(y_test)

In [10]:
input = Input((image_size, image_size, 3))


side_x1 = conv_block(input, 64)
x = MaxPooling2D(pool_size=(2, 2))(side_x1)

side_x2 = conv_block(x, 128)
x = MaxPooling2D(pool_size=(2, 2))(side_x2)

side_x3 = conv_block(x, 256)
x = MaxPooling2D(pool_size=(2, 2))(side_x3)

side_x4 = conv_block(x, 512)
x = MaxPooling2D(pool_size=(2, 2))(side_x4)

x = conv_block(x, 1024)

x = Conv2DTranspose(512, 2, strides=2, padding="same")(x)
x = Concatenate()([x, side_x4])
x = conv_block(x, 512)

x = Conv2DTranspose(256, 2, strides=2, padding="same")(x)
x = Concatenate()([x, side_x3])
x = conv_block(x, 256)

x = Conv2DTranspose(128, 2, strides=2, padding="same")(x)
x = Concatenate()([x, side_x2])
x = conv_block(x, 128)

x = Conv2DTranspose(64, 2, strides=2, padding="same")(x)
x = Concatenate()([x, side_x1])
x = conv_block(x, 64)


output = Conv2D(num_classes, 1, padding='same', activation='softmax')(x)
model = Model(input, output, name='SNN')

In [11]:
model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="pixel_accuracy")],
    )

In [ ]:
model.fit(
    x=X_train,
    y=y_train, 
    batch_size=batch_size, 
    epochs=epochs, 
    shuffle=True
)

Epoch 1/24


c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


6/6 ━━━━━━━━━━━━━━━━━━━━ 177s 25s/step - loss: 1.8798 - pixel_accuracy: 0.0543
Epoch 2/24
6/6 ━━━━━━━━━━━━━━━━━━━━ 178s 30s/step - loss: 0.8706 - pixel_accuracy: 0.1272
Epoch 3/24
2/6 ━━━━━━━━━━━━━━━━━━━━ 2:19 35s/step - loss: 0.5931 - pixel_accuracy: 0.1359

In [ ]:
image = sample_df[0, 'image']
pred_mask = model.predict(image)
pred_mask = tf.argmax(pred_mask, axis=-1)